# N160 · Shell文本流与管道

**目标：** 用真实文本工具完成逐行和按列处理。

**先修：** N006, N010, N023。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** stdin/stdout；管道；grep/awk/sed；sort/uniq；正则；locale；退出码。

**配套讲解来源：** [同名逐章意见](../../comment/160_shell_text_processing.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

前几章我们处理的是 DataFrame（结构化的表），这一章回到最原始的形态：**文本流**。Shell 管道里流淌的就是一行一行的纯文本，四个脚本解决四道 LeetCode Shell 题：数词频（192）、挑合法电话号码（193）、转置表格（194）、打印第十行（195）。所有脚本都从标准输入读、往标准输出写，这样它们才能像积木一样用 `|` 串起来；在 LeetCode 的文件式判题接口里，用 `< file.txt` 重定向喂给脚本就行。

给你一个具体到数字的例子（词频题）。输入两行文本：

```
the day is sunny the the
the sunny is is
```

问每个词出现几次，输出按"次数从多到少、并列按字典序"排。手算：the 出现 4 次（第一行 3 次 + 第二行 1 次），is 3 次，sunny 2 次，day 1 次。所以输出是：

```
the 4
is 3
sunny 2
day 1
```

再看电话题的"锚点"为什么重要：输入里有 `987-123-4567`（合法）和 `987-123-4567 `（末尾多一个空格，**不合法**）。如果只用"包含"式的模糊匹配，两个都会被选中；本章用"整行匹配"的正则，第二个被拒——部分像不等于整行是。

## 2. 基础知识：先读懂这些词

- **stdin / stdout（标准输入/输出）**：每个程序默认的两个管道口——stdin 是它读数据的地方，stdout 是它写结果的地方。脚本不写死文件名，从 stdin 读，才能被任意上游喂任意数据。
- **管道 `|`**：把左边程序的 stdout 接到右边程序的 stdin，文本像水流一样经过一节节处理。本章词频脚本内部就有一条：`awk` 数完数，管道送给 `sort` 排序。
- **awk**：逐行扫描的小型文本处理语言。每行自动按空白切成字段：`$1` 是第一个字段、`$2` 是第二个，`NF` 是本行字段数，`NR` 是当前行号。它有"每行做什么"和"END（扫完后做什么）"两段结构，还支持 `count[key]++` 这样的关联数组（键值对计数器）。
- **grep**：按模式筛行的工具（"含有这个模式的行留下"）。本章电话题改用 awk 的正则来做同一件事，效果等价。
- **sed（流编辑器）**：对文本流做编辑动作。本章只用了 `sed -n '10p'`：`-n` 表示"默认不打印"，`10p` 表示"第 10 行打印"，合起来就是"只打印第十行"。
- **sort / uniq**：`sort -k2,2nr` 意思是"按第 2 列、数值（n）、逆序（r）排"；`-k1,1` 是"再按第 1 列升序"。`uniq -c` 能把**已排序**的相邻重复行压缩计数（本章用 awk 的关联数组达到同样效果，uniq 没直接出场）。
- **正则与锚点 `^...$`**：`^` 匹配行首、`$` 匹配行尾，两个锚点夹住整个模式，才能保证"整行匹配"而不是"行内某段匹配"。`\(...\)` 在正则里表示分组，要匹配字面括号得写 `\(` `\)`。
- **locale（语言环境）**：排序规则跟语言环境走——不同 locale 对大小写、重音字符的排法不同。`export LC_ALL=C` 强制用最朴素的字节序（C locale）字典序，让排序结果在所有机器上一致。
- **退出码（exit code）**：程序结束时返回给操作系统的一个整数，0 表示成功，非 0 表示失败。本章转置脚本对"非矩形输入"返回 2，让调用方能**区分**"错误数据"和"正常的空结果"。
- **`set -euo pipefail`**：bash 的安全开关组合：`-e` 遇错即停、`-u` 用了未定义变量报错、`-o pipefail` 管道中任何一节失败都算整体失败。

## 3. 思路推导：从小例子开始

**Step 0：先立规矩（输入语法）。** “思路推导”部分 先固定四条契约：词由空白分隔且区分大小写（`The` 和 `the` 是两个词）；词频并列按 C locale 字典序分胜负；电话必须整行匹配；转置的输入必须是矩形的空白分隔表。这些"规矩"决定了后面每一步的写法。

**Step 1：词频 = 计数 + 双关键字排序。** 拿 `b a b a c` 手算：awk 逐行扫（这里只有一行），对每个字段执行 `count[$i]++`——扫完 b 后 count[b]=1，扫完 a 后 count[a]=1，再遇 b 时 count[b]=2……最终 count = {a:2, b:2, c:1}。END 块把每个"词 次数"对打出来（关联数组输出顺序不定）。接着 `sort -k2,2nr -k1,1` 接管：先按第 2 列（次数）数值逆序，再按第 1 列（词）字典序。手算排序：a 2 和 b 2 次数并列，按词升序 a 在前；c 1 殿后。输出 `a 2 / b 2 / c 1`。计数不变量是：**任意时刻，count 里存的恰好是"到目前为止已读入的文本"中每个词的出现次数**。

**Step 2：电话 = 锚点正则。** 合法格式只有两种：`xxx-xxx-xxxx` 或 `(xxx) xxx-xxxx`。正则 `^([0-9][0-9][0-9]-|\([0-9][0-9][0-9]\) )[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]$` 翻译过来是：行首开头，要么"三位数字加横杠"，要么"括号三位数字加空格"，接着三位数字加横杠、四位数字，行尾收束。手算测试集（见第六节）：六个候选只有 `987-123-4567`、`(123) 456-7890`、`000-000-0000` 过关，末尾带空格的和括号里只有两位的都被锚点挡下。

**Step 3：转置 = 记下 (行,列) 再按列吐出。** 输入 `name age / alice 21 / bob 23`（3 行 2 列）。awk 第一行先记 `width=NF`（列数=2）当标准；之后每行检查 `NF!=width`，列数不对就置 bad 标志并以退出码 2 离场——**宁可报错，不把残缺数据当空表**。正常路径把每个字段存进 `cell[NR,i]=$i`（第 NR 行第 i 列），扫完后 END 块换序输出：外层循环 i 从 1 到 width（遍历**列**），内层循环 j 从 1 到 NR（遍历原行），打印 `cell[j,i]`——即新行的第 i 行由原来的第 i 列从头到尾拼成。手算：新第 1 行 = cell[1,1], cell[2,1], cell[3,1] = `name alice bob`；新第 2 行 = `age 21 23`。`printf "%s%s", 字段, (j==NR ? "\n" : " ")` 表示每打一个字段补一个空格、打到最后一行换成换行。

**Step 4：第十行 = sed 一发入魂。** `sed -n '10p'` 只打印第 10 行：输入 1 到 11 共 11 行时输出 `10`；输入只有 2 行时没有第 10 行，输出空；空输入同样输出空。全程不把文件读进 Python，一条 sed 直接搞定。

## 4. 核心代码：word_frequency.sh

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```bash
#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk '{ for (i=1;i<=NF;i++) count[$i]++ } END { for (word in count) print word, count[word] }' | sort -k2,2nr -k1,1
```

### 逐段读懂代码

“分段实现”部分 是 Python 驱动：一个字典装四个脚本的源码，一个函数负责"落盘→用 bash 真跑→收回输出"。四个脚本都是真 `awk`/`sed`，不是模拟。四个脚本共享同一个三行头部——`#!/usr/bin/env bash` 指定解释器、`set -euo pipefail` 装上安全开关、`export LC_ALL=C` 钉死排序规则——这个"头部模板"本身就是 shell 脚本的工程惯例：让脚本在任何机器上行为一致。

**第 1 段：四个脚本本体（“运行示例”部分 以 Code 块原样展示过）。**

```bash
#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk '{ for (i=1;i<=NF;i++) count[$i]++ } END { for (word in count) print word, count[word] }' | sort -k2,2nr -k1,1
```

词频脚本。主块把本行每个字段在关联数组 `count` 里加一；END 块把"词 次数"逐对打印。输出交给管道右边的 `sort`：`-k2,2nr` 按次数数值逆序，`-k1,1` 并列时按词升序，C locale 保证字典序确定。

```bash
#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk '/^([0-9][0-9][0-9]-|\([0-9][0-9][0-9]\) )[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]$/ { print }'
```

电话脚本。awk 的 `/正则/ { print }` 就是"整行匹配正则就打印本行"——grep 的用法搬到 awk 上。`\(`、`\)` 转义后匹配字面括号，`^`、`$` 两端锚定整行。

```bash
#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk 'NR==1 { width=NF }
     NF!=width { bad=1; exit 2 }
     { for (i=1;i<=NF;i++) cell[NR,i]=$i }
     END { if (bad) exit 2;
           for (i=1;i<=width;i++) {
             for (j=1;j<=NR;j++) printf "%s%s", cell[j,i], (j==NR ? "\n" : " ")
           }
     }'
```

转置脚本，四段规则对应 Step 3：首行定宽度 `width`；列数不符就 `bad=1; exit 2`（END 里再 `exit 2` 兜底，确保非矩形输入一定以退出码 2 结束）；正常行把字段写进二维关联数组 `cell[行,列]`；END 双层循环按"先列后行"的顺序重排输出。注意 awk 的 `exit 2` 仍会执行 END 块，所以 END 开头要检查 bad。

```bash
#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
sed -n '10p'
```

第十行脚本。`-n` 关掉默认打印，`10p` 只对第 10 行执行打印动作，一行命令完成全部工作。

**第 2 段：Python 执行器。**

```python
def run_shell(name, text):
    """Execute the actual script on stdin in a temporary directory."""
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / name
        path.write_text(SHELL_SOURCES[name], encoding='utf-8')
        result = subprocess.run(['bash', str(path)], input=text, text=True, capture_output=True, check=True, timeout=10, env={**os.environ, 'LC_ALL': 'C'})
        return result.stdout
```

这就是本章的"测试"方式：在临时目录里把脚本源码写成真实文件，用 `subprocess.run` 启动 `bash` 执行它，`input=text` 把测试文本喂进脚本的 stdin，`capture_output=True` 收集 stdout，`check=True` 让非零退出码立刻变成 Python 异常（转置脚本的报错就是靠这个被测到的），`timeout=10` 防挂死，`env` 再钉一次 `LC_ALL=C` 排除宿主环境影响。返回值就是脚本的 stdout 文本，供 assert 直接比对。

## 5. 分段实现：按顺序运行

**本章接口：** `word_frequency.sh`、`valid_phone_numbers.sh`、`transpose_file.sh`、`tenth_line.sh`

### 导入本章使用的库

In [1]:
import os
import subprocess
import tempfile
from pathlib import Path

### SHELL_SOURCES

In [2]:
SHELL_SOURCES = {'word_frequency.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk '{ for (i=1;i<=NF;i++) count[$i]++ } END { for (word in count) print word, count[word] }' | sort -k2,2nr -k1,1\n", 'valid_phone_numbers.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk '/^([0-9][0-9][0-9]-|\\([0-9][0-9][0-9]\\) )[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]$/ { print }'\n", 'transpose_file.sh': '#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk \'NR==1 { width=NF }\n     NF!=width { bad=1; exit 2 }\n     { for (i=1;i<=NF;i++) cell[NR,i]=$i }\n     END { if (bad) exit 2;\n           for (i=1;i<=width;i++) {\n             for (j=1;j<=NR;j++) printf "%s%s", cell[j,i], (j==NR ? "\\n" : " ")\n           }\n     }\'\n', 'tenth_line.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nsed -n '10p'\n"}

### run_shell

In [3]:
def run_shell(name, text):
    """Execute the actual script on stdin in a temporary directory."""
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / name
        path.write_text(SHELL_SOURCES[name], encoding='utf-8')
        result = subprocess.run(['bash', str(path)], input=text, text=True, capture_output=True, check=True, timeout=10, env={**os.environ, 'LC_ALL': 'C'})
        return result.stdout

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from IPython.display import Code
for name in SHELL_SOURCES:
    display(Code(SHELL_SOURCES[name], language='bash'))
text = 'the day is sunny the the\nthe sunny is is\n'
show_table(['stage','text','lines'], [
 ['input',repr(text),len(text.splitlines())],
 ['frequency',repr(run_shell('word_frequency.sh',text)),len(run_shell('word_frequency.sh',text).splitlines())],
 ['transpose',repr(run_shell('transpose_file.sh','name age\nalice 21\nbob 23')),2]])

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk '{ for (i=1;i<=NF;i++) count[$i]++ } END { for (word in count) print word, count[word] }' | sort -k2,2nr -k1,1

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk '/^([0-9][0-9][0-9]-|\([0-9][0-9][0-9]\) )[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]$/ { print }'

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk 'NR==1 { width=NF }
     NF!=width { bad=1; exit 2 }
     { for (i=1;i<=NF;i++) cell[NR,i]=$i }
     END { if (bad) exit 2;
           for (i=1;i<=width;i++) {
             for (j=1;j<=NR;j++) printf "%s%s", cell[j,i], (j==NR ? "\n" : " ")
           }
     }'

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
sed -n '10p'

stage,text,lines
input,'the day is sunny the the\nthe sunny is is\n',2
frequency,'the 4\nis 3\nsunny 2\nday 1\n',4
transpose,'name alice bob\nage 21 23\n',2


## 7. 正确性、适用条件与复杂度

计数不变量是已读前缀中每个词的出现次数。转置的第 i 个输出行，按原行序列出全部 `cell[j,i]`，因此与原矩阵第 i 列一一对应。正则锚点保证是完整合法电话，而不是字符串中存在合法片段。

**代价：** 词频 O(L+u log u)，L 为文本长度、u 为不同词数。电话与第十行线性扫描；转置存储全部 r×c 个字段，时间、空间均 O(rc)。

### 展开理解

**为什么对？** 三条论证。词频的不变量一句话：awk 扫到任何位置，`count` 数组恰好记录"已读前缀"里每个词的次数——每读一个词就给对应键加一，不多不少；END 时全部文本恰好读完，所以计数完整。排序的双关键字恰好实现契约"次数降序、并列按词升序"，C locale 让"字典序"有唯一定义。电话正则两端有 `^` 和 `$`，意味着模式必须**覆盖整行**——像 `987-123-4567 `（尾空格）、`(12) 345-6789`（括号里少一位）这种"行内有合法片段"的行进不来；“正确性与复杂度”部分 的话说，锚点保证的是"完整合法电话"，不是"字符串中存在合法片段"。转置的正确性是构造性的：输出的第 i 行由 `cell[1,i], cell[2,i], ..., cell[NR,i]` 依次拼成，恰好是原矩阵的第 i 列从上到下，所以输出确实是原矩阵的转置；而"首行定宽、异宽报错"保证 `cell[j,i]` 这个格子对每个 j 都真实存在，不会拼出残行。非矩形返回非零退出码同样是正确性的一部分：错误数据要被当成**错误**上抛，而不是被静默当成空结果糊弄过去。

**复杂度怎么说？** 词频分两截：awk 扫描整段文本是 O(L)（L 是文本长度，比如 1 MB 的日志就是约一百万次字段访问），排序只排 u 个**不同**词，是 O(u log u)——注意排序的不是所有词而是去重后的词表，一篇文章 10 万个词但不同词往往只有几千，log 几千 ≈ 12，代价很小。电话和第十行都是对输入的一次线性扫描。转置要把 r×c 个字段全部存进 `cell` 数组再换序输出，时间和空间都是 O(rc)——一个 1000 行 × 100 列的表就是 10 万个字段的存取，量级温和但确实是"全量装内存"，特别大的表要想到这一点。另外，管道的可爱之处在于内存是"流过"的：`awk | sort` 两节各自只需要保存自己的工作集（awk 一行一行处理只需 O(u) 的计数表，sort 需要 O(u) 的词表），整段 1 GB 的原始文本从头到尾不需要作为一个整体待在内存里。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的验证方式是：用 `run_shell` 真跑脚本，拿 stdout 文本逐条 assert；唯一例外是转置的报错场景，用"期待 subprocess 异常 + 退出码为 2"来测。先把预期答案押在手心再往下看：词频 `b a b a c` 应得 `a 2 / b 2 / c 1`；六个电话应过三个；3 行 2 列转置应得 2 行 3 列；1 到 11 行取第十行应得 `10`。

1. `run_shell('word_frequency.sh','') == ''` —— **空输入边界**：没有词就没有输出，脚本不能报错。
2. `run_shell('word_frequency.sh','b a b a c') == 'a 2\nb 2\nc 1\n'` —— **正常值 + 并列次序**：a、b 同为 2 次，必须按字典序 a 在 b 前；这条同时在测"计数正确"和"双关键字排序"，也间接测了 awk 关联数组的任意输出顺序会被 sort 纠正。
3. `valid_phone_numbers.sh` 对六个号码的输出 == 三个合法号码 —— **正常值 + 反例群**：`123 456 7890`（无横杠）、`987-123-4567 `（尾随空格）、`(12) 345-6789`（括号内两位）全部必须被锚点拒绝，前两种格式各留一个正例，`000-000-0000` 补一个"全零但合法"的例子防你写"数字非零"之类的花活。
4. `run_shell('valid_phone_numbers.sh','bad') == ''` —— **无匹配边界**：一行都不合法就输出空。
5. `transpose_file.sh` 对 3 行 2 列 == `name alice bob\nage 21 23\n` —— **正常值**：手算见 Step 3，行变列、列变行。
6. `run_shell('transpose_file.sh','') == ''` —— **空输入边界**：空表转置还是空。
7. `run_shell('transpose_file.sh','only') == 'only\n'` —— **单行单列边界**：1×1 矩阵转置不变。
8. `try: run_shell('transpose_file.sh','a b\nc') except CalledProcessError as error: assert error.returncode == 2` —— **错误契约**：第二行只有 1 列（首行是 2 列），脚本必须以退出码 2 失败；如果它"成功"返回（把残表当结果），测试反而 raise。`a b\nc` 这个输入的巧妙在于：不报错的话输出会是什么都不确定，只有退出码 2 才是正确行为。
9. `tenth_line.sh` 三连：输入 `1\n2`（不足 10 行）输出空 —— **不足边界**；输入 1..11 共 11 行输出 `10` —— **正常值**；输入空串输出空 —— **空输入边界**。

In [5]:
assert run_shell('word_frequency.sh', '') == ''

assert run_shell('word_frequency.sh', 'b a b a c') == 'a 2\nb 2\nc 1\n'

phones = '987-123-4567\n(123) 456-7890\n123 456 7890\n987-123-4567 \n(12) 345-6789\n000-000-0000'

assert run_shell('valid_phone_numbers.sh', phones) == '987-123-4567\n(123) 456-7890\n000-000-0000\n'

assert run_shell('valid_phone_numbers.sh', 'bad') == ''

assert run_shell('transpose_file.sh', 'name age\nalice 21\nbob 23') == 'name alice bob\nage 21 23\n'

assert run_shell('transpose_file.sh', '') == ''

assert run_shell('transpose_file.sh', 'only') == 'only\n'

try:
    run_shell('transpose_file.sh', 'a b\nc')
except subprocess.CalledProcessError as error:
    assert error.returncode == 2
else:
    raise AssertionError('Ragged table was accepted')

assert run_shell('tenth_line.sh', '1\n2') == ''

lines = '\n'.join(map(str, range(1, 12)))

assert run_shell('tenth_line.sh', lines).strip() == '10'

assert run_shell('tenth_line.sh', '').strip() == ''

print('N160: 所有本章断言通过')

N160: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 处理不满10行文件。

**练习 2：** 说明sort和locale如何影响排序，不用Python重写题解。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（处理不满 10 行的文件）**：提示——先用本章脚本跑 `'1\n2'` 和 `'1\n...\n9'`，确认两种"不满 10 行"的输出都是空；再想 LeetCode 195 的判题机拿到空输出会怎么判。你可以尝试写一个变体：不足 10 行时输出特定提示或非零退出码（参考转置脚本的 `exit 2` 套路），并说清楚"输出空"和"报错"分别适合什么业务场景。边界手算示例：9 行输入时第 10 行不存在，`sed -n '10p'` 天然输出空，这算不算"把缺行当空行"？整理答案时建议列三行：恰好 10 行、不足 10 行、空文件，各写一个预期输出。
- **练习 2（说明 sort 和 locale 如何影响排序）**：提示——构造一个大小写混合的词表如 `Apple apple Banana banana`，分别在 `LC_ALL=C`、`LC_ALL=en_US.UTF-8`（如果本机支持）下跑 `sort`：C locale 按字节序所有大写字母排在小写前面（Apple, Banana, apple, banana），而某些 locale 会大小写穿插（Apple, apple, Banana, banana）。再跑一遍本章词频脚本（它钉死了 C）对比同输入不同环境的行为，解释为什么"结果要可复现就必须钉 locale"。全程只用 shell 工具，不许用 Python 重写题解。落笔时先写清楚三件事再动手：

- 输入词表原样列出，两种 locale 下的预期输出先手算。
- 实测命令原样记录（含环境变量前缀），输出贴在旁边。
- 一句话结论：并列词频的次序契约依赖 locale，不钉死就没有"确定"的答案。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

本章包含真实的 SQL／Shell／JavaScript／线程程序，Python 部分负责创建示例、调用运行时和断言。嵌入的程序源码也在下面完整显示；缺少相应运行时应安装依赖，不能用模拟结果替代。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
import os

import subprocess

import tempfile

from pathlib import Path

SHELL_SOURCES = {'word_frequency.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk '{ for (i=1;i<=NF;i++) count[$i]++ } END { for (word in count) print word, count[word] }' | sort -k2,2nr -k1,1\n", 'valid_phone_numbers.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk '/^([0-9][0-9][0-9]-|\\([0-9][0-9][0-9]\\) )[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]$/ { print }'\n", 'transpose_file.sh': '#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk \'NR==1 { width=NF }\n     NF!=width { bad=1; exit 2 }\n     { for (i=1;i<=NF;i++) cell[NR,i]=$i }\n     END { if (bad) exit 2;\n           for (i=1;i<=width;i++) {\n             for (j=1;j<=NR;j++) printf "%s%s", cell[j,i], (j==NR ? "\\n" : " ")\n           }\n     }\'\n', 'tenth_line.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nsed -n '10p'\n"}

def run_shell(name, text):
    """Execute the actual script on stdin in a temporary directory."""
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / name
        path.write_text(SHELL_SOURCES[name], encoding='utf-8')
        result = subprocess.run(['bash', str(path)], input=text, text=True, capture_output=True, check=True, timeout=10, env={**os.environ, 'LC_ALL': 'C'})
        return result.stdout

# 示例与回归测试
assert run_shell('word_frequency.sh', '') == ''

assert run_shell('word_frequency.sh', 'b a b a c') == 'a 2\nb 2\nc 1\n'

phones = '987-123-4567\n(123) 456-7890\n123 456 7890\n987-123-4567 \n(12) 345-6789\n000-000-0000'

assert run_shell('valid_phone_numbers.sh', phones) == '987-123-4567\n(123) 456-7890\n000-000-0000\n'

assert run_shell('valid_phone_numbers.sh', 'bad') == ''

assert run_shell('transpose_file.sh', 'name age\nalice 21\nbob 23') == 'name alice bob\nage 21 23\n'

assert run_shell('transpose_file.sh', '') == ''

assert run_shell('transpose_file.sh', 'only') == 'only\n'

try:
    run_shell('transpose_file.sh', 'a b\nc')
except subprocess.CalledProcessError as error:
    assert error.returncode == 2
else:
    raise AssertionError('Ragged table was accepted')

assert run_shell('tenth_line.sh', '1\n2') == ''

lines = '\n'.join(map(str, range(1, 12)))

assert run_shell('tenth_line.sh', lines).strip() == '10'

assert run_shell('tenth_line.sh', '').strip() == ''

print('N160: 所有本章断言通过')

N160: 所有本章断言通过


## 11. 代表题与迁移

- **192. Word Frequency**：练 awk 关联数组计数 + sort 双关键字排序，对应 `word_frequency.sh`，并列次序就是这题的隐藏考点。
- **193. Valid Phone Numbers**：练整行锚点正则（两种电话格式二选一），对应 `valid_phone_numbers.sh`。
- **194. Transpose File**：练 awk 二维存储 (行,列) + 换序输出，外加"非矩形输入报错"的防御性契约，对应 `transpose_file.sh`。
- **195. Tenth Line**：练 `sed -n '10p'` 精确选行与"不足 10 行输出空"的边界行为，对应 `tenth_line.sh`。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。